# 거시경제 지표에 따른 국내 이커머스 카테고리별 매출 탄력성 및 소비 전이 분석

- **용도:** 2026 국가데이터활용대회 공모전 참가 및 학생설계융합전공 졸업논문
  
- **분석 목적:** 거시경제 지표(물가, 금리, 유가)에 따른 국내 전자상거래 카테고리별 실질 거래액의 변동을 비교 및 예측하며, 소득 계층에 따른 소비 전이 현상을 파악
  
- **분석 활용 데이터:** KOSIS(온라인쇼핑동향조사, 품목별 소비자물가지수, 생활물가지수), ECOS(가계대출금리, 두바이유가, 경제심리지수), MDIS(가계동향조사)

- **2 Track 분석:**
    - Track 1. 22개 카테고리별 매출 탄력성 비교
    - Track 2. 소득 5분위별 소비 전이 효과 분석

In [ ]:
# Ignore the warnings
import warnings
# warnings.filterwarnings('always')
warnings.filterwarnings('ignore')

# System related and data input controls
import os

# Data manipulation and visualization
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.max_rows = 20
pd.options.display.max_columns = 20
import numpy as np
import requests
from functools import reduce
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing

# 시각화 한글 폰트 깨짐 방지 (Windows 기준)
from matplotlib import font_manager, rc
font_path = "C:/Windows/Fonts/malgun.ttf"
font = font_manager.FontProperties(fname=font_path).get_name()
rc('font', family=font)

# Modeling algorithms
# General
import statsmodels.api as sm
from scipy import stats

# Model selection
from sklearn.model_selection import train_test_split

# Evaluation metrics
from sklearn import metrics

# Import Library: 분석에 사용할 모듈 설치
!pip install --upgrade pip
!python -m pip install --user --upgrade pip

In [ ]:
df = pd.read_excel(r"C:\Users\sigtd\SJM\Data\온라인쇼핑동향조사_17.01~26.02.xlsx")

df.set_index('상품군별(1)', inplace=True)

df_transposed = df.T

date_strings = df_transposed.index.astype(str).str.replace('.', '')
df_transposed.index = pd.to_datetime(date_strings, format='%Y%m')

df_transposed.index.name = 'Date'

df_온라인거래액 = df_transposed.apply(pd.to_numeric, errors='coerce')

display(df_온라인거래액.head(3))
print(df_온라인거래액.info())

In [ ]:
df_cpi_raw = pd.read_excel(r"C:\Users\sigtd\SJM\Data\소비자물가지수_17.01~26.02.xlsx")
df_cpi_raw.set_index(df_cpi_raw.columns[0], inplace=True)
df_cpi_t = df_cpi_raw.T

cleaned_dates = df_cpi_t.index.str.replace(' 월', '').str.replace('.', '-')
df_cpi_t.index = pd.to_datetime(cleaned_dates)
df_cpi_t.index.name = 'Date'

df_CPI = df_cpi_t.apply(pd.to_numeric, errors='coerce')

display(df_CPI.head(3))
print(df_CPI.info())

## (종속변수 계산) 실질 온라인거래액
- 'KOSIS 온라인쇼핑동향조사'와 'KOSIS 소비자물가지수(CPI)' 데이터 활용
- '소비자물가지수(CPI)'의 458개 품목을 '온라인쇼핑동향조사'의 23개 카테고리와 매핑
- 매핑 완료 후 카테고리별로 명목 거래액을 대표 CPI로 나누어 물가 상승을 고려한 실질 거래액(종속변수) 계산
- **대표 CPI 합성 방식: 기하평균(Geometric Mean)** — CPI는 multiplicative 성격의 지수이므로 산술평균보다 기하평균이 적합하며, 미국 BLS도 lower-level CPI 합성에 기하평균(Jevons index)을 사용. 매핑 품목 수 차이(예: 음·식료품 70개 vs 사무·문구 4개)에 robust하고, 단일 품목의 극단적 가격 변동 영향이 작음. (Robustness check로 'median', 'arithmetic' 옵션도 함수 내 제공)

In [ ]:
### CPI와 온라인거래액 매핑
# 1. 컬럼명 추출
# '합계'를 제외한 23개의 온라인 쇼핑 카테고리 추출
online_categories = df_온라인거래액.columns.drop('합계').tolist()
# 458개의 CPI 품목 추출
cpi_items = df_CPI.columns.tolist()

# 2. 카테고리별 매핑 딕셔너리 뼈대 생성
# Key: 온라인 카테고리명 / Value: 대표 CPI 품목명(리스트 형태)
category_mapping = {category: [] for category in online_categories}

category_mapping['컴퓨터 및 주변기기'] = ['컴퓨터', '저장장치', '컴퓨터소모품']
category_mapping['가전·전자·통신기기'] = ['휴대전화기', 'TV', '영상음향기기', '휴대용멀티미디어기기', '전기밥솥', '가스레인지', '전자레인지', '전기레인지', '냉장고', '김치냉장고', '에어컨', '선풍기', '공기청정기', '세탁기', '의류건조기', '식기세척기', '청소기', '소형주방가전', '헤어드라이어']
category_mapping['서적'] = ['유아용학습교재', '초등학교학습서', '중학교학습서', '고등학교학습서', '대학교재', '서적', '신문']
category_mapping['사무·문구'] = ['종이문구', '기타문구', '필기구', '회화용구']
category_mapping['의복'] = ['남자외의', '남자상의', '남자하의', '남자내의', '여자외의', '원피스', '여자상의', '여자하의', '여자내의', '점퍼', '티셔츠', '스웨터', '청바지', '운동복', '등산복', '유아동복', '양말']
category_mapping['신발'] = ['구두', '운동화', '실내화', '아동화']
category_mapping['가방'] = ['가방', '핸드백', '지갑', '우산']
category_mapping['패션용품 및 액세서리'] = ['손목시계', '장신구', '모자', '장갑', '선글라스']
category_mapping['스포츠·레저용품'] = ['자전거', '헬스기구', '레저용품', '운동용품']
category_mapping['화장품'] = ['기초화장품', '기능성화장품', '색조화장품', '모발염색약']
category_mapping['아동·유아용품'] = ['유모차', '장난감', '종이기저귀']

# 건강기능식품, 홍삼, 비타민제 등을 모두 음·식료품으로 통합
category_mapping['음·식료품'] = ['밀가루', '국수', '라면', '당면', '두부', '시리얼', '부침가루', '케이크', '빵', '떡', '파스타면', '소시지', '햄및베이컨', '기타육류가공품', '오징어채', '북어채', '어묵', '맛살', '수산물통조림', '젓갈', '우유', '분유', '치즈', '발효유', '참기름', '식용유', '과일가공품', '단무지', '맛김', '초콜릿', '사탕', '껌', '아이스크림', '비스킷', '스낵과자', '파이', '설탕', '잼', '물엿', '소금', '간장', '된장', '양념소스', '고추장', '카레', '식초', '드레싱', '혼합조미료', '스프', '이유식', '김치', '밑반찬', '냉동식품', '즉석식품', '편의점도시락', '삼각김밥', '커피', '차', '주스', '두유', '생수', '기능성음료', '탄산음료', '기타음료', '소주', '과실주', '맥주', '막걸리', '양주', '약주', '홍삼', '건강기능식품', '유산균', '비타민제']
category_mapping['농축수산물'] = ['쌀', '현미', '찹쌀', '보리쌀', '콩', '땅콩', '혼식곡', '배추', '상추', '시금치', '양배추', '미나리', '깻잎', '부추', '무', '열무', '당근', '감자', '고구마', '도라지', '콩나물', '버섯', '오이', '풋고추', '호박', '가지', '토마토', '파', '양파', '마늘', '브로콜리', '고사리', '파프리카', '생강', '사과', '배', '복숭아', '포도', '밤', '감', '귤', '오렌지', '참외', '수박', '딸기', '바나나', '키위', '블루베리', '망고', '체리', '아보카도', '파인애플', '아몬드', '고춧가루', '참깨', '인삼', '꿀', '국산쇠고기', '수입쇠고기', '돼지고기', '닭고기', '달걀', '갈치', '명태', '조기', '고등어', '오징어', '게', '굴', '조개', '전복', '새우', '마른멸치', '마른오징어', '낙지', '김', '미역']
category_mapping['생활용품'] = ['화초', '보온매트', '면도기', '침구', '커튼', '주택수선재료', '식기', '컵', '솥', '프라이팬', '냄비', '수저', '밀폐용기', '부엌용용구', '건전지', '소형가사용품', '세탁세제', '섬유유연제', '전구', '부엌용세제', '청소용세제', '살충제', '가정용비닐용품', '키친타월', '방향제', '습기제거제', '마스크', '반창고', '칫솔', '치약', '비누', '화장지', '구강세정제', '생리대', '샴푸', '바디워시']
category_mapping['자동차 및 자동차용품'] = ['소형승용차', '중형승용차', '대형승용차', '경승용차', '다목적승용차', '수입승용차', '전기동력차', '자동차용품', '자동차타이어', '블랙박스']
category_mapping['가구'] = ['장롱', '침대', '거실장', '소파', '책상', '의자', '식탁', '싱크대']

# 오프라인 미용 서비스 제외 (사료, 용품 등 실물 위주)
category_mapping['애완용품'] = ['반려동물용품'] 

# 교통/숙박 예약 위주로 남김 (시내버스료, 통행료 등 오프라인 성격 강한 항목 삭제)
category_mapping['여행 및 교통서비스'] = ['국제항공료', '국내항공료', '여객선료', '국내단체여행비', '해외단체여행비', '호텔숙박료', '여관숙박료', '콘도이용료', '휴양시설이용료', '열차료', '시외버스료', '승용차임차료']

# 온라인 예매가 중심이 되는 항목만 남김 (당구장, 목욕탕 등 오프라인 결제 항목 삭제)
category_mapping['문화 및 레저서비스'] = ['영화관람료', '공연예술관람료', '운동경기관람료', '관람시설이용료']

# 배달 비중이 높은 외식 항목 중심으로 재편성
category_mapping['음식서비스'] = ['치킨', '피자', '햄버거', '자장면', '짬뽕', '탕수육', '볶음밥', '돈가스', '김밥', '떡볶이', '쌀국수', '냉면', '칼국수', '생선초밥', '도시락', '커피(외식)']

# 렌탈 및 이사 등 용역서비스 중심 (공과금, 수리비 등 삭제)
category_mapping['기타서비스'] = ['가전제품렌탈비', '건강기기렌탈비', '이삿짐운송료', '사진서비스료', '가사도우미료', '택배이용료']

# 담배, 연료 등 온라인 거래 불가/희박 품목 삭제
category_mapping['기타'] = ['의료측정기', '보청기', '치료재료', '안경', '콘택트렌즈', '악기', '원예용품', '보일러', '감기약', '진통제', '소화제', '위장약', '진해거담제', '소염진통제', '피부질환제', '치과구강용약', '가정용비닐용품']

In [ ]:
# 3. 실질 거래액(최종 종속변수 파생) 산출 함수 — 기하평균 기반 robust 합성
def calculate_real_volume(df_sales, df_cpi, mapping_dict, method='geometric'):
    """
    온라인 명목거래액을 대표 CPI로 나누어 실질 거래액(종속변수)을 계산

    [대표 CPI 합성 방법 — 카테고리별 매핑된 N개 품목 CPI를 단일 대표값으로 합성]
      • 'geometric'  (기본) : 기하평균 — CPI는 multiplicative(비율) 성격의 지수이므로
                              산술평균보다 기하평균이 통계적으로 적합. 미국 BLS도 lower-level
                              CPI 합성에 기하평균(Jevons index)을 사용. 품목 수 차이에 robust
                              하고, 한 품목의 극단적 가격 급등(예: 농산물 spike)이 전체 대표값을
                              왜곡하는 영향이 산술평균보다 작다.
      • 'median'             : 중앙값 — outlier에 가장 robust (robustness check용)
      • 'arithmetic'         : 산술평균 — 단순 평균 (비교/검증용)

    [Robustness check] 본 분석은 'geometric'을 기본으로 사용하되, 결과의 강건성을
    위해 'median' 및 'arithmetic'으로도 동일 분석을 반복해 결론이 일관되는지 확인 권장.
    """
    df_real = pd.DataFrame(index=df_sales.index)

    # '합계' 컬럼 제외 (종속변수가 아닌 보조 정보)
    sales_cols = df_sales.columns.drop('합계') if '합계' in df_sales.columns else df_sales.columns

    for category in sales_cols:
        mapped_items = mapping_dict.get(category, [])
        if not mapped_items:
            df_real[category] = np.nan
            continue

        # df_CPI에 실제 존재하는 컬럼만 필터링 (오타·누락 방지)
        valid_cpi_cols = [c for c in mapped_items if c in df_cpi.columns]
        if not valid_cpi_cols:
            print(f"⚠️ '{category}': 매핑된 {mapped_items}는 df_CPI에 존재하지 않는 컬럼입니다.")
            df_real[category] = np.nan
            continue

        cpi_subset = df_cpi[valid_cpi_cols]

        # 대표 CPI 합성 — method 옵션에 따라
        if method == 'geometric':
            # 기하평균 = exp(log값들의 산술평균)
            representative_cpi = np.exp(np.log(cpi_subset).mean(axis=1))
        elif method == 'median':
            representative_cpi = cpi_subset.median(axis=1)
        elif method == 'arithmetic':
            representative_cpi = cpi_subset.mean(axis=1)
        else:
            raise ValueError(f"지원하지 않는 method: {method}")

        # 실질 거래액 = 명목 거래액 / (CPI 지수 / 100)  ※ 2020년=100 기준
        df_real[category] = df_sales[category] / (representative_cpi / 100)

    return df_real

In [ ]:
df_실질거래액 = calculate_real_volume(df_온라인거래액, df_CPI, category_mapping)
display(df_실질거래액.head())
print(df_실질거래액.info())

In [ ]:
df_실질거래액 = df_실질거래액.drop('이쿠폰서비스', axis=1)

In [ ]:
df_실질거래액

## (독립변수 추가) 실질 가계대출금리
- 'KOSIS 소비자물가지수(CPI)'와 'ECOS 예금은행 가계대출금리(잔액 기준)' 데이터 활용
- CPI의 총지수를 활용해 '(올해 총지수 - 작년 총지수) / 작년 총지수 × 100' 식으로 작년대비 물가상승률(YoY)을 계산
- 종속변수가 실질 거래액이므로 소비자의 실질적 구매력을 대변하는 '실질금리'만 사용 (명목 가계대출금리 - 물가상승률 = 실질 가계대출금리)

In [ ]:
# df_CPI에서 총지수를 이용해 물가상승률 계산
# periods=12는 '12개월 전(즉, 작년 같은 달)'과 비교하라는 뜻입니다.
df_CPI['물가상승률(YoY)'] = df_CPI['총지수'].pct_change(periods=12) * 100

# 17년도 물가상승률은 16년도 데이터가 없으므로 직접 기입
values_2017 = [2.241, 2.083, 2.277, 1.956, 2.004, 1.806, 2.167, 2.489, 2.000, 1.762, 1.153, 1.407]

df_CPI.loc['2017-01-01':'2017-12-01', '물가상승률(YoY)'] = values_2017

# 결과 확인
display(df_CPI[['총지수', '물가상승률(YoY)']])

In [ ]:
# ==============================================================================
# 1. ECOS 금리 데이터 불러오기 및 추출
# ==============================================================================
file_path_rate = r"C:\Users\sigtd\SJM\Data\예금은행 가계대출금리(잔액 기준)_17.01~26.02.xlsx"
df_rate_raw = pd.read_excel(file_path_rate)

# '계정항목' 컬럼에서 '가계대출'이 포함된 행만 필터링 (마이너스통장 포함 행)
# ECOS 파일 구조상 정확한 텍스트 매칭을 위해 str.contains를 사용합니다.
df_rate_filtered = df_rate_raw[df_rate_raw['계정항목'].str.contains('가계대출', na=False)]

# ==============================================================================
# 2. 시계열 전처리 (가로형 -> 세로형 및 날짜 인덱스 설정)
# ==============================================================================
# 가로로 퍼진 날짜(2017/01, 2017/02...)를 'Date'라는 하나의 열로 내립니다(Melt).
df_rate_melted = df_rate_filtered.melt(id_vars=['계정항목'], var_name='Date', value_name='명목_가계대출금리')

# '2017/01' 형태의 문자열을 완벽한 시계열(Datetime) 객체인 '2017-01-01'로 변환합니다.
df_rate_melted['Date'] = pd.to_datetime(df_rate_melted['Date'], format='%Y/%m')

# 날짜를 인덱스로 지정하고, 금리 데이터를 계산 가능한 숫자형으로 변환합니다.
df_rate_melted.set_index('Date', inplace=True)
df_rate_melted['명목_가계대출금리'] = pd.to_numeric(df_rate_melted['명목_가계대출금리'], errors='coerce')

# 최종적으로 병합에 사용할 금리 컬럼만 남깁니다.
df_rate_final = df_rate_melted[['명목_가계대출금리']]

# ==============================================================================
# 3. [핵심] 기존 데이터 결합 및 실질금리(최종 독립변수) 파생변수 생성
# ==============================================================================
# 기존에 물가상승률이 포함된 df_CPI 데이터와 날짜(Index)를 기준으로 병합합니다.
# how='inner'를 사용하여 두 데이터의 날짜가 완벽히 일치하는 구간만 남깁니다.
df_master = pd.merge(df_CPI, df_rate_final, left_index=True, right_index=True, how='inner')

# 피셔 방정식(Fisher Equation) 적용: 실질금리 = 명목금리 - 물가상승률
df_master['실질_가계대출금리'] = df_master['명목_가계대출금리'] - df_master['물가상승률(YoY)']

# ==============================================================================
# 4. 최종 결과 확인
# ==============================================================================
print("[데이터 결합 및 파생변수 생성 완료]")
print("현재 데이터 크기:", df_master.shape)
print("\n[최종 독립변수 데이터 미리보기]")
display(df_master[['물가상승률(YoY)', '명목_가계대출금리', '실질_가계대출금리']])

In [ ]:
df_master_final = df_실질거래액.join(df_master[['실질_가계대출금리']], how='inner')

## (독립변수 추가) 생활물가상승률
- 'KOSIS 생활물가지수' 데이터 활용
- 소비자물가지수(CPI)는 총 458개 품목 대상으로 국가 전체 인플레이션을 보여주지만, 생활물가지수는 구입 빈도가 높고 지출 비중이 높은 144개 필수 생필품만 모아 놓은 지수. 소비자 예산 제약과 체감경기 악화를 대변하는데 CPI보다 생활물가지수가 더 적합함. 풍선효과(선택재/사치재에서 필수재로의 품목 전이 현상)를 증명하는 핵심 변수임.
- 물가상승률과 마찬가지로 '(올해 총지수 - 작년 총지수) / 작년 총지수 × 100' 식으로 작년대비 물가상승률(YoY)을 계산함.

In [ ]:
# 1. 데이터 로딩
df_living_raw = pd.read_excel(r"C:\Users\sigtd\SJM\Data\생활물가지수_16.01~26.02.xlsx")

# 2. 행렬 전치 및 시계열 인덱스 변환 (Wide -> Long 형태의 변형)
# '품목별' 열을 인덱스로 세팅하고 행과 열을 전치(.T)합니다.
df_living_raw.set_index('품목별', inplace=True)
df_living_cpi = df_living_raw.T

# 인덱스(날짜 문자열) 전처리: '2016.01' -> '2016-01-01' 형태로 시계열 변환
# 점(.)을 없애고 날짜형 객체로 바꿔줍니다.
df_living_cpi.index = df_living_cpi.index.str.replace('.', '')
df_living_cpi.index = pd.to_datetime(df_living_cpi.index, format='%Y%m')
df_living_cpi.index.name = 'Date'

# 안전한 계산을 위해 데이터 타입을 모두 실수형(float)으로 변환합니다.
df_living_cpi = df_living_cpi.apply(pd.to_numeric, errors='coerce')

# 3. 생활물가상승률(YoY) 파생변수 생성
# 첫 번째 열('생활물가지수' 총지수)을 지정하여 전년 동월 대비 증감률을 계산합니다.
target_col = df_living_cpi.columns[0]
df_living_cpi['생활물가상승률(YoY)'] = df_living_cpi[target_col].pct_change(periods=12) * 100

# 결합을 위해 필요한 파생변수 컬럼만 떼어냅니다.
df_living_feature = df_living_cpi[['생활물가상승률(YoY)']]

# 4. 기존 실질거래액 데이터셋과 최종 결합 (Merge)
# how='inner'를 통해 양쪽 데이터에 모두 존재하는 2017년 1월 ~ 2026년 2월 구간만 남깁니다.
df_master_final = df_master_final.join(df_living_feature, how='inner')

In [ ]:
# 결과 확인
print("데이터 크기:", df_master_final.shape)
display(df_master_final.head())

## (독립변수 추가) 유가상승률
- 'ECOS 국제상품가격_두바이유' 데이터 활용
- 한국은 필요한 원유의 약 70%를 중동 지역에서 수입하므로 미국 텍사스에서 생산되는 WTI나 유럽의 Brent유 가격변동보다 두바이유의 가격변동이 국내 정유사의 도입 단가에 직접적이고 절대적인 영향을 미침
- 전년동기대비 증감률로 유가상승률(YoY)을 독립변수로써 사용함

In [ ]:
# 1. 유가상승률 데이터 로드
file_oil = r"C:\Users\sigtd\SJM\Data\유가상승률_17.01~26.02.xlsx"
df_oil_raw = pd.read_excel(file_oil)

# 2. Wide -> Long 변환
id_col_oil = df_oil_raw.columns[0]
df_oil_melted = df_oil_raw.melt(id_vars=[id_col_oil], var_name='Date', value_name='유가상승률(YoY)')

# 3. 날짜(Date) 전처리 및 인덱스 설정
df_oil_melted['Date'] = pd.to_datetime(df_oil_melted['Date'].astype(str).str.replace('/', '-'), format='%Y-%m')
df_oil_melted.set_index('Date', inplace=True)

# 4. 숫자형 변환 및 최종 피처 추출
df_oil_melted['유가상승률(YoY)'] = pd.to_numeric(df_oil_melted['유가상승률(YoY)'], errors='coerce')
df_oil_feature = df_oil_melted[['유가상승률(YoY)']].dropna()

# 결과 확인
print("✅ 유가상승률 전처리 완료")
display(df_oil_feature)

## (독립변수 추가) 경제심리지수(ESI) 순환변동치
- 지금까지의 독립변수(실질 가계대출금리, 생활물가상승률, 유가상승률)는 거시경제 영향으로 예산제약에 따른 물리적 타격이라면, 경제심리지수(ESI)는 심리적 타격을 대변함
- 기업과 소비자 모두를 포함한 민간의 경제상황에 대한 심리를 종합적으로 파악하기 위하여 BSI 및 CSI 지수를 합성하여 경제심리지수(ESI : Economic Sentiment Index)를 만듦
- 경제심리지수의 원계열, 순환변동치 중 순환변동치 데이터를 사용함
- '순환변동치'는 원계열에서 일시적인 충격이나 계절적 요인을 수학적으로 다 발라내고, "진짜 현재 경기가 바닥을 향해 가고 있는지, 회복세에 있는지"를 보여주는 순수한 트렌드 지표. 장기적인 거시경제 충격을 분석하는 논문의 목적에는 순환변동치가 완벽하게 부합함

In [ ]:
# 1. ESI 데이터 로드
file_esi = r"C:\Users\sigtd\SJM\Data\경제심리지수(ESI)_순환변동치_17.01~26.02.xlsx"
df_esi_raw = pd.read_excel(file_esi)

# 2. Wide -> Long 변환 (첫 번째 열이 '계정항목' 등의 이름표라고 가정)
# df_esi_raw.columns[0]을 사용하여 첫 번째 열의 이름이 무엇이든 유연하게 대처합니다.
id_col_esi = df_esi_raw.columns[0]
df_esi_melted = df_esi_raw.melt(id_vars=[id_col_esi], var_name='Date', value_name='ESI_순환변동치')

# 3. 날짜(Date) 전처리 및 인덱스 설정
# '2017/01' 형태를 '2017-01-01' 형태의 시계열 객체로 변환
df_esi_melted['Date'] = pd.to_datetime(df_esi_melted['Date'].astype(str).str.replace('/', '-'), format='%Y-%m')
df_esi_melted.set_index('Date', inplace=True)

# 4. 숫자형 변환 및 최종 피처 추출
df_esi_melted['ESI_순환변동치'] = pd.to_numeric(df_esi_melted['ESI_순환변동치'], errors='coerce')
df_esi_feature = df_esi_melted[['ESI_순환변동치']].dropna()

# 결과 확인
print("✅ ESI 순환변동치 전처리 완료")
display(df_esi_feature)

In [ ]:
# 유가상승률 및 경제심리지수 독립변수 결합
df_master_final = df_master_final.join([df_oil_feature, df_esi_feature], how='inner')
display(df_master_final)
print(df_master_final.info())

## 파생변수 추가 및 나머지 전처리

**파생변수 추가 전략 및 추가 전처리 개요**

**A. 종속변수 변환**
- **로그변환된 실질거래액** : 분산 안정화 + ARDL 계수의 탄력성(elasticity) 해석 가능

**B. 거시 충격 통제 변수**
- **COVID-19 더미** : 2020.03~2022.04 사회적 거리두기 구간 통제 (구조변화 보정)

**C. 거시변수 동학(dynamic) 파생변수**
- **시차변수 (lag 1, 3, 6개월)** : 거시 충격이 즉각 반영되지 않고 시차를 두고 발현되는 효과 포착
- **차분변수 (Δ, 전월대비 변화량)** : 금리 인상/인하 같은 변화량 자체의 충격 측정
- **변동성 (6개월 rolling std)** : 불확실성 자체가 예비적 저축 동기를 자극해 소비를 위축시키는 효과 (uncertainty shock)

**D. 풍선효과·소비전이 가설 검증용 합성 변수**
- **거시 스트레스 종합지수** : 4개 거시변수 z-score 합성 (실질금리·생활물가·유가는 +, ESI는 −) → 클러스터링·시각화에서 단일 압박 지표로 사용
- **금리×생활물가 교호작용** : 가계 실질소득 이중 압박을 직접 측정 (Track 1 풍선효과 가설의 핵심)

**E. 계절성·추세 통제**
- **월별 더미 (m_2~m_12)** : 명절·블랙프라이데이 등 월별 고정효과 통제 (1월=기준범주)
- **시간 추세 (trend)** : 이커머스 시장 자체의 장기 성장 추세 통제

### (A) 종속변수 로그 변환
— 분산 안정화 + ARDL 계수의 탄력성(elasticity) 해석 가능

In [ ]:
# 카테고리(종속변수) 컬럼만 추출 — 거시 독립변수는 제외
exog_vars     = ['실질_가계대출금리', '생활물가상승률(YoY)', '유가상승률(YoY)', 'ESI_순환변동치']
category_cols = [c for c in df_master_final.columns if c not in exog_vars]

# 자연로그 변환 (실질거래액은 항상 양수이므로 안전하게 적용 가능)
df_log = df_master_final.copy()
for col in category_cols:
    df_log[f'log_{col}'] = np.log(df_master_final[col])

print(f"✅ 로그변환 완료: {len(category_cols)}개 카테고리 → log_<카테고리명>")
display(df_log[[f'log_{c}' for c in category_cols[:5]]].head(3))

### (B) COVID-19 더미변수
- 분석 구간(2017.01~2026.02)은 팬데믹(2020.03~2022.04)이라는 거대한 구조변화를 포함 → 통제하지 않으면 ARDL 계수 추정이 왜곡됨
- 한국 정부의 사회적 거리두기 시행~해제 구간(2020.03 ~ 2022.04)을 외생 더미변수로 모형에 투입
- 논문(Gupta & Varshney, 2023)이 글로벌 금융위기 더미를 활용한 사례와 동일한 robustness 보강 장치

In [ ]:
# ==============================================================================
# (B) COVID-19 더미변수 추가 — 구조변화(structural break) 통제
# ==============================================================================
# 한국 사회적 거리두기 시행 구간: 2020.03 ~ 2022.04 (2022.04.18 사회적 거리두기 전면 해제)
df_log['covid_dummy'] = (
    (df_log.index >= '2020-03-01') & (df_log.index <= '2022-04-30')
).astype(int)

print(f"✅ COVID-19 더미 추가 완료")
print(f"   해당 월(=1) 개수: {df_log['covid_dummy'].sum()}개월")
print(f"   비해당 월(=0)   : {(df_log['covid_dummy']==0).sum()}개월")

### (C) 시차변수(lag) 및 차분변수(diff)
- **시차변수** : 거시 충격이 즉각 반영되지 않고 시차를 두고 발현되는 효과를 포착. ARDL 자체가 시차구조를 모델링하지만, 사전 EDA(상관관계 매트릭스, 시차 산점도)와 단순 회귀 비교에서도 유용. 1, 3, 6개월 시차를 추가
- **차분변수** : 변수의 수준(level)이 아니라 `변화량(Δ)` 자체가 소비에 미치는 충격(예: 기준금리 인상 직후 lump-sum shock)을 측정

In [ ]:
# ==============================================================================
# (C) 거시변수 시차변수(lag) 및 차분변수(diff) 추가
# ==============================================================================
# 거시변수 4개 정의 (이후 셀에서 재사용)
macro_vars = ['실질_가계대출금리', '생활물가상승률(YoY)', '유가상승률(YoY)', 'ESI_순환변동치']

# (C-1) 시차변수: 거시 충격이 시차를 두고 소비에 반영되는 효과 포착 (1, 3, 6개월)
for lag in [1, 3, 6]:
    for v in macro_vars:
        df_log[f'{v}_lag{lag}'] = df_log[v].shift(lag)

# (C-2) 차분변수: 변화량(Δ) 자체의 충격 측정 (예: 금리 인상 shock)
for v in macro_vars:
    df_log[f'{v}_diff'] = df_log[v].diff(1)

n_lag_added  = len(macro_vars) * 3
n_diff_added = len(macro_vars)
print(f"✅ 시차변수 {n_lag_added}개 + 차분변수 {n_diff_added}개 추가 완료")
print(f"   ※ 초기 1~6개월 행에 NaN 발생 — ARDL 추정 시 자동 처리됨")

### (D) 변동성·합성지수·교호작용
- **변동성 (6개월 rolling std)** : 거시변수의 변동 폭이 클수록 가계는 미래 불확실성에 대비해 소비를 줄이고 저축을 늘림 (예비적 저축 동기, precautionary saving). 평균값(level) 외에 변동성(uncertainty) 자체가 별도의 충격 채널
- **거시 스트레스 종합지수** : 4개 거시변수를 z-score로 표준화한 뒤, 가계 부담을 키우는 방향(실질금리·생활물가·유가는 +, ESI 순환변동치는 −)으로 합산한 단일 종합 압박 지표. 카테고리 클러스터링·시계열 시각화에서 `한 줄 요약 변수`로 활용
- **금리 × 생활물가 교호작용** : 실질금리 상승(이자 부담↑) + 생활물가 상승(필수재 비용↑)이 동시에 발생하면 가처분소득에 이중 압박 → 풍선효과(선택재→필수재 전이)의 강도를 직접 측정하는 변수

In [ ]:
# ==============================================================================
# (D-1) 거시변수 변동성 (6개월 rolling std) — 불확실성 충격
# (D-2) 거시 스트레스 종합지수 — 4개 거시변수 z-score 합성
# (D-3) 교호작용 — 실질금리 × 생활물가상승률 (실질소득 이중압박)
# ==============================================================================

# (D-1) 6개월 이동표준편차
window = 6
for v in macro_vars:
    df_log[f'{v}_vol{window}m'] = df_log[v].rolling(window=window, min_periods=3).std()

# (D-2) z-score 표준화 후 합성지수 산출
#   - 실질금리·생활물가·유가는 ↑일수록 가계 부담 ↑ (부호 +)
#   - ESI 순환변동치는 ↑일수록 경기개선 (부호 −, 즉 부담 ↓)
z = (df_log[macro_vars] - df_log[macro_vars].mean()) / df_log[macro_vars].std()
df_log['거시스트레스지수'] = (
    z['실질_가계대출금리']
    + z['생활물가상승률(YoY)']
    + z['유가상승률(YoY)']
    - z['ESI_순환변동치']
)

# (D-3) 교호작용: 금리 × 생활물가 (실질소득 이중압박 효과 — 풍선효과 핵심 변수)
df_log['금리X생활물가'] = df_log['실질_가계대출금리'] * df_log['생활물가상승률(YoY)']

vol_cols = [f'{v}_vol{window}m' for v in macro_vars]
print(f"✅ 변동성 {len(vol_cols)}개 + 거시스트레스지수 + 금리X생활물가 추가 완료")
display(df_log[vol_cols + ['거시스트레스지수', '금리X생활물가']].tail(3))

### (E) 계절성·추세 통제 변수
- **월별 더미** : 명절(설/추석), 블랙프라이데이, 연말 시즌 등 월 단위 고정효과 통제. 1월(m_1)을 기준범주로 하여 m_2 ~ m_12까지 11개 더미 생성
- **시간 추세 (trend)** : 이커머스 시장 자체의 장기 성장 추세를 통제하지 않으면 거시변수와 상관관계가 잡혀 spurious regression 위험. 1, 2, 3, ... 형태의 선형 추세 추가

In [ ]:
# ==============================================================================
# (E) 계절성 더미(월별) + 시간 추세
# ==============================================================================
# (E-1) 월별 더미 — 1월을 기준범주로 두고 m_2 ~ m_12 (총 11개) 생성
month_dummies = pd.get_dummies(
    pd.Series(df_log.index.month, index=df_log.index),
    prefix='m', drop_first=True
).astype(int)
df_log = df_log.join(month_dummies)

# (E-2) 시간 추세 — 이커머스 시장 자체의 장기 성장 추세 통제
df_log['trend'] = np.arange(1, len(df_log) + 1)

print(f"✅ 월별 더미 {month_dummies.shape[1]}개 + 시간 추세 1개 추가 완료")
display(df_log[list(month_dummies.columns) + ['trend']].head(3))

In [ ]:
# ==============================================================================
# 최종 데이터셋 점검 (ARDL 입력 데이터 확정)
# ==============================================================================
# 컬럼 그룹별 카운트 (해석 편의 + 발표자료용 스냅샷)
n_cat       = len(category_cols)
n_log       = len([c for c in df_log.columns if c.startswith('log_')])
n_macro     = len(macro_vars)
n_lag       = len([c for c in df_log.columns if '_lag' in c])
n_diff      = len([c for c in df_log.columns if c.endswith('_diff')])
n_vol       = len([c for c in df_log.columns if '_vol' in c])
n_synth     = 2   # 거시스트레스지수, 금리X생활물가
n_covid     = 1
n_month     = len([c for c in df_log.columns if c.startswith('m_')])
n_trend     = 1

print("=" * 60)
print("[ 최종 ARDL 입력 데이터셋 구성 ]")
print("=" * 60)
print(f"종속변수(원본 실질거래액)        : {n_cat:>3}개")
print(f"종속변수(로그변환)              : {n_log:>3}개")
print(f"거시 원변수                     : {n_macro:>3}개")
print(f"거시 시차변수(1, 3, 6개월)      : {n_lag:>3}개")
print(f"거시 차분변수(Δ)                : {n_diff:>3}개")
print(f"거시 변동성(6m rolling std)     : {n_vol:>3}개")
print(f"합성변수(스트레스지수, 교호작용)  : {n_synth:>3}개")
print(f"COVID-19 더미                   : {n_covid:>3}개")
print(f"월별 더미(m_2 ~ m_12)           : {n_month:>3}개")
print(f"시간 추세                       : {n_trend:>3}개")
print("-" * 60)
print(f"전체 컬럼 수                    : {df_log.shape[1]:>3}개")
print(f"전체 관측치                     : {df_log.shape[0]:>3}개")

# NaN 발생 컬럼 안내 (lag/rolling 계산 특성상 초기 시점에서 NaN 발생 → ARDL 추정 시 자동 drop)
print("\n[ NaN 발생 컬럼 — lag/rolling 계산 특성상 초기 시점에서 발생, ARDL 추정 시 자동 처리됨 ]")
nan_summary = df_log.isna().sum()
nan_summary = nan_summary[nan_summary > 0].sort_values(ascending=False)
display(nan_summary.to_frame('NaN 개수'))

print("\n✅ 전처리 단계 종료 — 다음 단계: 정상성/공적분 검정 → ARDL 22회 추정")

# 분석(Track 1): 카테고리별 매출 탄력성 비교
- 목적: 금리, 물가, 유가 등 거시 충격이 22개 상품군 중 어디를 가장 아프게 때리는가? 필수재(버티기) vs 선택재(무너짐)의 풍선효과 증명.

## EDA 1 — 시계열 패턴 (22 카테고리 + 4 거시변수)
- COVID-19 구간(2020.03~2022.04)을 음영으로 표시해 구조변화 시각 확인
- 카테고리별 패턴 분기를 한눈에 보는 것이 Track 1 스토리의 출발점

In [ ]:
# ==============================================================================
# (EDA-1a) 22개 카테고리 실질거래액 시계열 — 패턴 분기 시각화
# ==============================================================================
fig, axes = plt.subplots(6, 4, figsize=(20, 22))
axes = axes.flatten()

for i, cat in enumerate(category_cols):
    ax = axes[i]
    ax.plot(df_log.index, df_log[cat], color='#1f77b4', linewidth=1.2)
    ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2022-04-30'),
               color='red', alpha=0.10)
    ax.set_title(cat, fontsize=10)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(True, alpha=0.3)

# 빈 subplot 숨기기 (24 - 22 = 2)
for j in range(len(category_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('22개 카테고리 실질거래액 시계열 (붉은 음영: COVID-19 구간)', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# (EDA-1b) 거시변수 4종 시계열 — COVID 구간과 함께 보기
# ==============================================================================
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
axes = axes.flatten()

for i, v in enumerate(macro_vars):
    ax = axes[i]
    ax.plot(df_log.index, df_log[v], color='#d62728', linewidth=1.5)
    ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2022-04-30'),
               color='gray', alpha=0.15)
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_title(v, fontsize=12)
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, alpha=0.3)

plt.suptitle('거시변수 4종 시계열 (회색 음영: COVID-19 구간)', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()


## EDA 2 — 기술통계 + 변동성(CV) 비교
- 변동계수(CV = 표준편차/평균)가 큰 카테고리 = 거시 충격에 민감할 가능성
- 이후 ARDL 탄력성과 비교해 "변동성 큰 카테고리가 실제로 탄력적이었는지" 검증

In [ ]:
# ==============================================================================
# (EDA-2) 22개 카테고리 기술통계 + 변동계수(CV) 시각화
# ==============================================================================
desc = df_log[category_cols].describe().T[['mean', 'std', 'min', 'max']]
desc['CV(%)'] = (desc['std'] / desc['mean'] * 100).round(2)
desc = desc.sort_values('CV(%)', ascending=False)
desc.columns = ['평균', '표준편차', '최소', '최대', 'CV(%)']

print("[ 22개 카테고리 기술통계 — CV 내림차순 ]")
display(desc.style.format({'평균': '{:,.0f}', '표준편차': '{:,.0f}',
                            '최소': '{:,.0f}', '최대': '{:,.0f}', 'CV(%)': '{:.2f}'}))

# CV 시각화
fig, ax = plt.subplots(figsize=(12, 7))
desc['CV(%)'].plot(kind='barh', ax=ax, color='#2ca02c')
ax.set_xlabel('변동계수 CV(%)')
ax.set_title('카테고리별 변동계수 (CV ↑일수록 거시 충격에 민감할 가능성)', fontsize=13)
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## EDA 3 — 종속↔독립 상관관계 (동시점 + 시차)
- **동시점 heatmap (22 × 4)** : 거시변수별 영향 받는 카테고리 한눈에 — 발표용 핵심 자료
- **시차 상관 (lag 0~6)** : 거시 충격이 어느 시차에서 가장 강하게 반영되는지 → ARDL 최적 시차 선택의 사전 근거

In [ ]:
# ==============================================================================
# (EDA-3a) 22 카테고리 × 4 거시변수 동시점 상관관계 heatmap
# ==============================================================================
log_cols = [f'log_{c}' for c in category_cols]
corr_inst = df_log[log_cols + macro_vars].corr().loc[log_cols, macro_vars]
corr_inst.index = [c.replace('log_', '') for c in corr_inst.index]

fig, ax = plt.subplots(figsize=(8, 11))
sns.heatmap(corr_inst, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, cbar_kws={'label': 'Pearson 상관계수'},
            linewidths=0.5, ax=ax)
ax.set_title('22 카테고리(log 실질거래액) × 4 거시변수 동시점 상관관계', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# (EDA-3b) 시차 상관관계 — 거시변수(t-k) vs log 거래액(t), lag 0~6
# ==============================================================================
max_lag = 6
fig, axes = plt.subplots(1, 4, figsize=(22, 10), sharey=True)

for i, v in enumerate(macro_vars):
    lag_corrs = pd.DataFrame(index=category_cols,
                              columns=range(max_lag + 1), dtype=float)
    for lag in range(max_lag + 1):
        x = df_log[v].shift(lag)
        for cat in category_cols:
            lag_corrs.loc[cat, lag] = df_log[f'log_{cat}'].corr(x)

    sns.heatmap(lag_corrs.astype(float), annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, cbar=(i == 3),
                cbar_kws={'label': '상관'} if i == 3 else None,
                linewidths=0.3, ax=axes[i])
    axes[i].set_title(v, fontsize=11)
    axes[i].set_xlabel('lag (개월)')
    if i > 0:
        axes[i].set_ylabel('')

plt.suptitle('시차 상관관계 — 거시변수(t−k) vs log 실질거래액(t)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## EDA 4 — ADF(Augmented Dickey-Fuller) 단위근 검정
- **목적** : ARDL 적합 전 정상성(stationarity) 사전 진단
- **귀무가설(H0)** : 단위근 존재 (비정상)
- **결정 기준** : p-value < 0.05 → H0 기각 → 정상(I(0))
- 비정상 변수는 1차 차분 후 재검정 (I(1) 여부 확인) — Bound test 가능 조건

In [ ]:
# ==============================================================================
# (EDA-4) ADF 단위근 검정 — 22 카테고리(log) + 4 거시변수
# ==============================================================================
from statsmodels.tsa.stattools import adfuller

def adf_test(series, name):
    s = series.dropna()
    res = adfuller(s, autolag='AIC')
    return {'변수': name, 'ADF통계량': res[0], 'p-value': res[1],
            '결론': '정상(I(0))' if res[1] < 0.05 else '비정상'}

# Level (원자료) 검정
results = []
for cat in category_cols:
    results.append(adf_test(df_log[f'log_{cat}'], f'log_{cat}'))
for v in macro_vars:
    results.append(adf_test(df_log[v], v))

adf_df = pd.DataFrame(results)
print("[ ADF 검정 결과 — Level(원자료) ]")
display(adf_df.style.format({'ADF통계량': '{:.3f}', 'p-value': '{:.4f}'}))

# 비정상 분류 변수에 대한 1차 차분 ADF 재검정
non_stationary = adf_df[adf_df['결론'] == '비정상']['변수'].tolist()
diff_results = []
for var in non_stationary:
    res = adfuller(df_log[var].diff().dropna(), autolag='AIC')
    diff_results.append({'변수': var, 'ADF통계량': res[0], 'p-value': res[1],
                          '결론': '정상(I(1))' if res[1] < 0.05 else '여전히 비정상'})

if diff_results:
    print(f"\n[ 비정상 변수 {len(diff_results)}개 — 1차 차분 후 ADF 재검정 ]")
    display(pd.DataFrame(diff_results).style.format({'ADF통계량': '{:.3f}', 'p-value': '{:.4f}'}))


## EDA 5 — 풍선효과 가설 사전 시각화
- **가설** : 생활물가 ↑ 시기 → 필수재(농축수산물·음식료품·생활용품) 소비 유지/증가, 선택재(가구·자동차·여행·문화) 감소 → '풍선효과'
- **방법** : 생활물가상승률을 4분위(Q1=낮음 ~ Q4=높음)로 분할 → 각 구간 카테고리별 평균 log 거래액 비교 → **Q4 − Q1 차이**가 클수록 거시 충격에 민감
- ARDL 추정 전 `눈으로 한 번` 가설 방향성을 확인하는 단계

In [ ]:
# ==============================================================================
# (EDA-5) 풍선효과 가설 — 생활물가상승률 4분위 구간별 카테고리 평균 비교
# ==============================================================================
df_eda = df_log[['생활물가상승률(YoY)'] + [f'log_{c}' for c in category_cols]].dropna().copy()
df_eda['생활물가_분위'] = pd.qcut(df_eda['생활물가상승률(YoY)'], q=4,
                                labels=['Q1(낮음)', 'Q2', 'Q3', 'Q4(높음)'])

log_cat_cols = [f'log_{c}' for c in category_cols]
mean_by_q = df_eda.groupby('생활물가_분위', observed=True)[log_cat_cols].mean()

# Q4 - Q1: 고물가 시기 - 저물가 시기 카테고리별 평균 차이
delta = (mean_by_q.loc['Q4(높음)'] - mean_by_q.loc['Q1(낮음)']).sort_values()
delta.index = [c.replace('log_', '') for c in delta.index]

# 시각화
fig, ax = plt.subplots(figsize=(11, 8))
colors = ['#d62728' if v < 0 else '#2ca02c' for v in delta.values]
delta.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Δ log(실질거래액)  =  Q4(고물가) − Q1(저물가)')
ax.set_title('생활물가 고/저 시기 카테고리별 평균 거래액 차이\n'
             '(녹색 = 고물가에 ↑, 풍선효과의 수혜 / 붉은색 = 고물가에 ↓, 타격)',
             fontsize=12)
plt.tight_layout()
plt.show()

print("\n[ Q4(고물가) − Q1(저물가) 차이 정렬 — 풍선효과 방향성 확인 ]")
display(delta.to_frame('Δ log(거래액)').style.format('{:+.3f}'))
